# Fine-tuning con Hard Negatives (Celulares)

Este notebook entrena el modelo agregando 28 imágenes de celulares de costado como **hard negatives** para reducir falsos positivos de `knife`.

**Estrategia:**
1. Cargar dataset aumentado desde Drive
2. Subir 28 celulares desde local (VSCode)
3. Integrarlos al training (NO al test)
4. Fine-tuning desde checkpoint existente

In [1]:
# Instalar dependencias
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 26.6 MB/s eta 0:00:0000:01


In [2]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Ir al repositorio en Drive
%cd /content/drive/MyDrive/procesamiento-imagenes

/content/drive/MyDrive/procesamiento-imagenes


In [24]:
# Copiar dataset aumentado a disco local (más rápido para GPU)
!cp "/content/drive/MyDrive/procesamiento-imagenes/dataset_augmented.zip" /content/dataset_augmented.zip
!unzip -q /content/dataset_augmented.zip -d /content/dataset_local

## Subir Hard Negatives (Celulares)

**IMPORTANTE:** Ejecutá esto desde VSCode. Los archivos se suben desde tu máquina local.

Ruta local: `/home/gbenito/universidad/procesamiento-imagenes-unlu/data/hard_negatives_celulares/`

In [10]:
# Crear carpeta para hard negatives en Colab
!mkdir -p /content/hard_negatives_celulares/images
!mkdir -p /content/hard_negatives_celulares/xmls

In [ ]:
# Subir archivos desde VSCode (clic en carpeta o drag & drop)
# O usar este código para upload programático:
from google.colab import files
import os

print("📤 Subiendo imágenes de celulares...")
# Opcional: comentá esto si ya subiste manualmente
uploaded_imgs = files.upload()
for filename in uploaded_imgs:
    !mv "{filename}" /content/hard_negatives/images/

print("\n📤 Subiendo XMLs de celulares...")
uploaded_xmls = files.upload()
for filename in uploaded_xmls:
    !mv "{filename}" /content/hard_negatives/xmls/

In [12]:
!cp "/content/drive/MyDrive/procesamiento-imagenes/hard_negatives_celulares.zip" /content/hard_negatives_celulares.zip
!unzip -q /content/hard_negatives_celulares.zip -d /content/

In [13]:
# Verificar que se subieron los 28
!echo "Imágenes:"
!ls /content/hard_negatives/images | wc -l
!echo "XMLs:"
!ls /content/hard_negatives/xmls | wc -l

Imágenes:
28
XMLs:
28


## Integrar Hard Negatives al Training

Los copiamos al dataset de training (NO al test) para que el modelo aprenda que "celular ≠ cuchillo".

In [27]:
# Copiar al dataset de training
!cp /content/hard_negatives/images/* /content/dataset_local/dataset_augmented/images/
!cp /content/hard_negatives/xmls/* /content/dataset_local/dataset_augmented/xmls/

print("✅ Hard negatives integrados al training")

✅ Hard negatives integrados al training


In [28]:
# Verificar cantidad total ahora
!echo "Total imágenes training:"
!ls /content/dataset_local/dataset_augmented/images | wc -l
!echo "Total XMLs training:"
!ls /content/dataset_local/dataset_augmented/xmls | wc -l

Total imágenes training:
29959
Total XMLs training:
29959


## Fine-tuning

Ajustamos hiperparámetros para fine-tuning:
- **LR bajo:** `1e-5` para no destruir conocimiento previo
- **Pocas épocas:** 10-15 con early stopping
- **Batch pequeño:** 4-8 para convergencia suave
- **Desde checkpoint:** continuamos desde el mejor modelo anterior

In [21]:
# Fine-tuning desde checkpoint anterior
# AJUSTÁ la ruta del checkpoint según tu modelo actual
!python3 pipeline_entrenamiento.py \
  --skip-stages split augment \
  --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth \
  --augmented-images /content/hard_negatives/images \
  --augmented-xmls /content/hard_negatives/xmls \
  --epochs 5 \
  --batch-size 6 \
  --enhance \
  --amp \
  --save-every 1 \
  --patience 5 \
  --output-dir results_finetuning_negatives
# --lr "1e-5" \


🎯 PIPELINE DE ENTRENAMIENTO COMPLETO
📅 Inicio: 2026-02-23 13:47:38
📂 Dataset original: dataset/images
📂 Salida: results_finetuning_negatives

⏭️  Saltando etapa: split

⏭️  Saltando etapa: augment

🚀 ETAPA: 3. Train Model
📝 Comando: python3 train_fasterrcnn_light.py --images-dir /content/hard_negatives/images --xml-dir /content/hard_negatives/xmls --output-dir results_finetuning_negatives --epochs 5 --batch-size 6 --lr 0.0001 --save-every 1 --patience 5 --enhance --amp --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth

🔧 Configuración: Dispositivo=cuda, Tamaño Imagen=(320, 320)
GPU: Tesla T4
✅ ImageEnhancer inicializado para mejorar calidad de imágenes
📦 Parseando 23 archivos XML (usando CPU multi-core)...
✅ 19 muestras válidas encontradas.              
✅ ImageEnhancer inicializado para mejorar calidad de imágenes
📦 Parseando 5 archivos XML (usando CPU multi-core)...
✅ 4 muestras válidas encontradas.              
Downloading: "https://download.py

## Evaluación

Después del fine-tuning, verificá:
1. **FP de knife bajaron?** (métrica clave)
2. **Recall de knife se mantuvo?** (no debe bajar más de 2-3%)
3. **Matriz de confusión:** verificar que casos "celular→knife" bajaron

In [22]:
# Ver resultados
!ls -lh results_finetuning_negatives/

total 84K
-rw------- 1 root root   54 Feb 23 13:48 classes.json
-rw------- 1 root root  624 Feb 23 13:48 pipeline_results.json
-rw------- 1 root root  77K Feb 23 13:48 training_history.png
-rw------- 1 root root 4.7K Feb 23 13:48 training_log.json


In [23]:
!cat results_finetuning_negatives/*.json

{
  "classes": {
    "knife": 1,
    "pistol": 2
  }
}{
  "pipeline_start": "2026-02-23T13:47:38.869363",
  "pipeline_end": "2026-02-23T13:48:06.839492",
  "stages": {
    "3. Train Model": {
      "success": true,
      "duration_sec": 27.968611001968384,
      "command": "python3 train_fasterrcnn_light.py --images-dir /content/hard_negatives/images --xml-dir /content/hard_negatives/xmls --output-dir results_finetuning_negatives --epochs 5 --batch-size 6 --lr 0.0001 --save-every 1 --patience 5 --enhance --amp --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth"
    }
  },
  "success": false,
  "total_duration_sec": 27.97012734413147
}[
  {
    "epoch": 1,
    "train_loss": 0.6116282853326293,
    "val_loss": 0.5919122953306545,
    "map": 0.5135780572891235,
    "map_50": 0.8790756464004517,
    "map_75": 0.498940110206604,
    "total_time_sec": 2224.6609168052673
  },
  {
    "epoch": 2,
    "train_loss": 0.5832012231621658,
    "val_loss": 0.669963

In [ ]:
# Copiar resultados a Drive
!cp -r results_finetuning_negatives /content/drive/MyDrive/procesamiento-imagenes/
print("✅ Resultados guardados en Drive")

In [ ]:
# Evaluar en el test set (con seguridad para no contaminar)
!python3 test_light_model.py \
  --model results_finetuning_negatives/best_model.pth \
  --images-dir /content/dataset_testing/images \
  --xml-dir /content/dataset_testing/xmls \
  --output-dir test_results_finetuning_negatives \
  --confidence 0.5

In [ ]:
# Fine-tuning correcto: usar DATASET COMPLETO (donde ya integraste los 28)
!python3 train_fasterrcnn_light.py \
  --images-dir /content/dataset_local/dataset_augmented/images \
  --xml-dir /content/dataset_local/dataset_augmented/xmls \
  --output-dir results_finetuning_negatives \
  --epochs 30 \
  --batch-size 6 \
  --lr 1e-5 \
  --save-every 1 \
  --patience 5 \
  --enhance \
  --amp \
  --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth

🔧 Configuración: Dispositivo=cuda, Tamaño Imagen=(320, 320)
GPU: Tesla T4
✅ ImageEnhancer inicializado para mejorar calidad de imágenes
📦 Parseando 25465 archivos XML (usando CPU multi-core)...
✅ 25460 muestras válidas encontradas.                            
✅ ImageEnhancer inicializado para mejorar calidad de imágenes
📦 Parseando 4494 archivos XML (usando CPU multi-core)...
✅ 4494 muestras válidas encontradas.                           
📥 Cargando checkpoint: /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth
✅ Checkpoint cargado (Época 20, mAP: 0.8304)
🔄 Reanudando desde época 21

🚀 INICIANDO ENTRENAMIENTO...
📊 Épocas: 21 → 30
💾 Guardando checkpoints cada 1 épocas
⏸️  Early stopping con paciencia = 5 épocas

--- Epoch 21/30 (RAM: 1.61GB) ---
🏋️ Training: 100% 4244/4244 [18:22<00:00,  3.85batch/s, loss=0.3786] 
✅ Epoch 21 | Train Loss: 0.3822 | Val Loss: 0.4077 | mAP: 0.8243 | Time: 1441.4s
📦 Checkpoint guardado: checkpoint_epoch_21.pth

--- Epoch 22/30 (RA

## Prueba Manual

Probá el nuevo modelo con un video de celular para ver si bajaron los FPs.

In [ ]:
# Ejemplo de inferencia con el nuevo modelo
# (ajustá según tu pipeline de inferencia)
# !python3 src/weapon_detection/inference/run_pipeline.py \
#   --input test_celular.mp4 \
#   --weapon-model results_finetuning_negatives/best_model.pth